# 🌊 Normalizing Flows

This notebook builds deep intuition for:
1. **Change of variables** — how transforming space changes probability
2. **The Jacobian** — measuring how transformations stretch and squeeze
3. **Flow composition** — building complexity from simple steps
4. **Coupling layers** — the trick that makes flows practical
5. **Exact likelihood** — why flows are special among generative models
6. **From simple to complex** — morphing a Gaussian into anything
7. **Visualizing the flow** — watching distributions transform step by step

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 1. The Magic of Change of Variables

Imagine you have a ball of clay shaped like a bell curve (Gaussian). You can **stretch**, **squeeze**,
**twist**, and **bend** it into any shape you want. But the total amount of clay stays the same —
probability must sum to 1.

The **change of variables formula** tells us exactly how the density changes under transformation:

$$p_Y(y) = p_X(f^{-1}(y)) \cdot \left|\frac{df^{-1}}{dy}\right|$$

- When $f$ **stretches** a region: density **decreases** (same probability spread over larger area)
- When $f$ **compresses** a region: density **increases** (same probability packed into smaller area)

Let's see this in action.

In [ ]:
# 1D Change of Variables: transforming a Gaussian

x = np.linspace(-4, 4, 1000)
p_x = norm.pdf(x)  # Standard Gaussian

# Three transformations
transforms = [
    {
        'name': 'y = 2x + 1 (linear stretch)',
        'f': lambda x: 2*x + 1,
        'f_inv': lambda y: (y - 1) / 2,
        'f_inv_deriv': lambda y: 0.5 * np.ones_like(y),
        'y_range': np.linspace(-8, 10, 1000),
    },
    {
        'name': 'y = sinh(x) (nonlinear stretch)',
        'f': np.sinh,
        'f_inv': np.arcsinh,
        'f_inv_deriv': lambda y: 1.0 / np.sqrt(y**2 + 1),
        'y_range': np.linspace(-8, 8, 1000),
    },
    {
        'name': 'y = exp(x) (exponential)',
        'f': np.exp,
        'f_inv': np.log,
        'f_inv_deriv': lambda y: 1.0 / y,
        'y_range': np.linspace(0.01, 8, 1000),
    },
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, t in enumerate(transforms):
    ax = axes[i]
    y = t['y_range']
    
    # Change of variables formula
    x_inv = t['f_inv'](y)
    p_y = norm.pdf(x_inv) * np.abs(t['f_inv_deriv'](y))
    
    # Also verify with samples
    samples_x = np.random.randn(50000)
    samples_y = t['f'](samples_x)
    
    ax.hist(samples_y, bins=100, density=True, alpha=0.3, color='steelblue', label='Samples')
    ax.plot(y, p_y, 'r-', linewidth=2, label='Theory')
    ax.set_title(t['name'])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Change of Variables: Transforming N(0,1)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("🔑 Key Insight: the formula gives EXACT densities — no approximation!")
print("This is what makes normalizing flows special: exact likelihood computation.")

## 2. The Jacobian — Measuring How Space Stretches

In 2D+, the Jacobian matrix $J = \frac{\partial f}{\partial x}$ captures how a transformation distorts space locally.

Its **determinant** $|\det(J)|$ tells us the local **volume change factor**:
- $|\det(J)| > 1$ → space is **stretching** → density **decreases**
- $|\det(J)| < 1$ → space is **compressing** → density **increases**
- $|\det(J)| = 1$ → volume-preserving

Let's visualize this by watching how a grid of points deforms.

In [ ]:
# Visualizing 2D transformations and their Jacobians

def make_grid(n=20, lim=2.5):
    t = np.linspace(-lim, lim, n)
    xx, yy = np.meshgrid(t, t)
    return np.column_stack([xx.ravel(), yy.ravel()]), n

def plot_transform(ax, pts_before, pts_after, n, det_J, title):
    """Plot deformed grid colored by Jacobian determinant."""
    # Draw grid lines
    for i in range(n):
        idx_h = np.arange(i*n, (i+1)*n)
        idx_v = np.arange(i, n*n, n)
        ax.plot(pts_after[idx_h, 0], pts_after[idx_h, 1], 'gray', alpha=0.2, lw=0.5)
        ax.plot(pts_after[idx_v, 0], pts_after[idx_v, 1], 'gray', alpha=0.2, lw=0.5)
    
    sc = ax.scatter(pts_after[:, 0], pts_after[:, 1], c=det_J, cmap='RdYlBu_r',
                    s=15, vmin=0.2, vmax=3, alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal')
    return sc

grid, n = make_grid(25, 2.5)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Transform 1: Rotation (volume-preserving)
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
grid_rot = grid @ R.T
det_rot = np.ones(len(grid))  # det = 1
sc = plot_transform(axes[0], grid, grid_rot, n, det_rot, f'Rotation (det=1 everywhere)')

# Transform 2: Nonlinear stretch
def swirl(z, strength=0.5):
    r = np.sqrt(z[:, 0]**2 + z[:, 1]**2)
    angle = strength * r
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    out = np.column_stack([
        z[:, 0] * cos_a - z[:, 1] * sin_a,
        z[:, 0] * sin_a + z[:, 1] * cos_a
    ])
    return out

grid_swirl = swirl(grid, 0.8)
# Numerical Jacobian determinant
eps = 1e-4
det_swirl = np.zeros(len(grid))
for k in range(len(grid)):
    J = np.zeros((2, 2))
    for d in range(2):
        pt_plus = grid[k].copy()
        pt_plus[d] += eps
        pt_minus = grid[k].copy()
        pt_minus[d] -= eps
        J[:, d] = (swirl(pt_plus.reshape(1,-1))[0] - swirl(pt_minus.reshape(1,-1))[0]) / (2*eps)
    det_swirl[k] = np.abs(np.linalg.det(J))

sc = plot_transform(axes[1], grid, grid_swirl, n, det_swirl, 'Swirl (det varies)')

# Transform 3: Squeeze one axis, stretch another
def squeeze(z):
    return np.column_stack([z[:, 0] * 2, z[:, 1] * 0.5])

grid_sq = squeeze(grid)
det_sq = np.ones(len(grid))  # det = 2 * 0.5 = 1
sc = plot_transform(axes[2], grid, grid_sq, n, det_sq, 'Squeeze (det=1, area preserved)')

# Transform 4: Nonlinear coupling-like
def coupling_transform(z):
    out = z.copy()
    out[:, 1] = z[:, 1] * np.exp(0.5 * np.sin(z[:, 0] * 2)) + 0.5 * np.cos(z[:, 0] * 3)
    return out

grid_coup = coupling_transform(grid)
det_coup = np.exp(0.5 * np.sin(grid[:, 0] * 2))  # analytical for coupling
sc = plot_transform(axes[3], grid, grid_coup, n, det_coup, 'Coupling layer (det=exp(s(x₁)))')

plt.colorbar(sc, ax=axes, shrink=0.8, label='|det(J)|')
plt.suptitle('How Different Transformations Deform 2D Space', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("🔑 The Jacobian determinant measures LOCAL volume change.")
print("Red regions = stretching (density decreases)")
print("Blue regions = compressing (density increases)")

## 3. Flow Composition — Stacking Simple Transforms

The power of normalizing flows comes from **composing** many simple transformations:

$$f = f_K \circ f_{K-1} \circ \cdots \circ f_1$$

The log-likelihood decomposes as a sum:

$$\log p(\mathbf{x}) = \log p_0(\mathbf{z}_0) - \sum_{k=1}^K \log|\det(J_k)|$$

Each simple step makes a small change. Together, they can transform a Gaussian into any distribution.

Think of it like **sculpting**: each chisel stroke is simple, but many strokes create a masterpiece.

In [ ]:
# Watching a Gaussian transform step by step

np.random.seed(42)

# Define a sequence of simple 2D transformations
def layer1(z):  # Rotate
    a = 0.3
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    return z @ R.T

def layer2(z):  # Scale + shift
    out = z.copy()
    out[:, 0] = z[:, 0] * 1.5 + 0.5
    return out

def layer3(z):  # Coupling: bend y based on x
    out = z.copy()
    out[:, 1] = z[:, 1] + 0.8 * np.sin(z[:, 0] * 1.5)
    return out

def layer4(z):  # Coupling: scale y based on x
    out = z.copy()
    out[:, 1] = z[:, 1] * np.exp(0.3 * np.tanh(z[:, 0]))
    return out

def layer5(z):  # Another coupling
    out = z.copy()
    out[:, 0] = z[:, 0] + 0.5 * np.tanh(z[:, 1] * 2)
    return out

def layer6(z):  # Final rotation
    a = -0.5
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    return z @ R.T

all_layers = [layer1, layer2, layer3, layer4, layer5, layer6]
layer_names = ['Rotate', 'Scale+Shift', 'Bend (coupling)', 
               'Scale (coupling)', 'Shift (coupling)', 'Rotate back']

# Start with Gaussian samples
z = np.random.randn(5000, 2)

fig, axes = plt.subplots(1, 7, figsize=(24, 3.5))

# Plot initial
axes[0].scatter(z[:, 0], z[:, 1], s=1, alpha=0.3, c='steelblue')
axes[0].set_title('z₀ ~ N(0,I)', fontsize=10)
axes[0].set_xlim(-5, 5)
axes[0].set_ylim(-5, 5)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Apply each layer
for i, (layer, name) in enumerate(zip(all_layers, layer_names)):
    z = layer(z)
    ax = axes[i + 1]
    ax.scatter(z[:, 0], z[:, 1], s=1, alpha=0.3, c='coral')
    ax.set_title(f'After: {name}', fontsize=9)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Normalizing Flow: Gaussian → Complex Distribution (Step by Step)', fontsize=14, y=1.08)
plt.tight_layout()
plt.show()

print("Each transformation is simple and invertible.")
print("Together, they sculpt the Gaussian into a complex shape!")
print("\n🔑 Composition is the key: simple × simple × ... × simple = complex")

## 4. Coupling Layers — The Trick That Makes Flows Practical

Computing $\det(J)$ is $O(n^3)$ in general — too expensive for high dimensions.

**Coupling layers** (from RealNVP) solve this brilliantly:

**Split** the input: $\mathbf{x} = [\mathbf{x}_1, \mathbf{x}_2]$

**Transform:**
- $\mathbf{y}_1 = \mathbf{x}_1$ (keep unchanged!)
- $\mathbf{y}_2 = \mathbf{x}_2 \odot \exp(s(\mathbf{x}_1)) + t(\mathbf{x}_1)$

where $s$ and $t$ are arbitrary neural networks.

**Why this is brilliant:**
1. The Jacobian is **triangular** → $\det(J) = \exp(\sum s_i(\mathbf{x}_1))$ — just $O(n)$!
2. The inverse is **trivial**: $\mathbf{x}_2 = (\mathbf{y}_2 - t(\mathbf{y}_1)) \odot \exp(-s(\mathbf{y}_1))$
3. $s$ and $t$ can be **any** neural network — no invertibility constraints
4. Alternate which half is transformed → all dimensions get updated

In [ ]:
# Coupling layers in action: building a flow from scratch

class AffineCouplingLayer:
    """Affine coupling layer: y1=x1, y2=x2*exp(s(x1))+t(x1)"""
    
    def __init__(self, mask_first=True, scale=0.5):
        self.mask_first = mask_first  # which half to keep fixed
        self.scale = scale
        # Simple parametric s and t functions
        self.a_s = np.random.randn() * scale
        self.b_s = np.random.randn() * scale
        self.c_s = np.random.randn() * scale
        self.a_t = np.random.randn() * scale
        self.b_t = np.random.randn() * scale
        self.c_t = np.random.randn() * scale
    
    def s(self, x):
        return self.a_s * np.tanh(self.b_s * x + self.c_s)
    
    def t(self, x):
        return self.a_t * np.sin(self.b_t * x + self.c_t)
    
    def forward(self, z):
        y = z.copy()
        if self.mask_first:  # fix dim 0, transform dim 1
            y[:, 1] = z[:, 1] * np.exp(self.s(z[:, 0])) + self.t(z[:, 0])
            log_det = self.s(z[:, 0])
        else:  # fix dim 1, transform dim 0
            y[:, 0] = z[:, 0] * np.exp(self.s(z[:, 1])) + self.t(z[:, 1])
            log_det = self.s(z[:, 1])
        return y, log_det
    
    def inverse(self, y):
        x = y.copy()
        if self.mask_first:
            x[:, 1] = (y[:, 1] - self.t(y[:, 0])) * np.exp(-self.s(y[:, 0]))
        else:
            x[:, 0] = (y[:, 0] - self.t(y[:, 1])) * np.exp(-self.s(y[:, 1]))
        return x

# Build a flow with alternating coupling layers
np.random.seed(42)
n_layers = 8
flow_layers = [AffineCouplingLayer(mask_first=(i % 2 == 0), scale=0.8) 
               for i in range(n_layers)]

# Push a Gaussian through the flow
z = np.random.randn(5000, 2)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes[0, 0].scatter(z[:, 0], z[:, 1], s=1, alpha=0.3, c='steelblue')
axes[0, 0].set_title('Base: N(0,I)')
axes[0, 0].set_aspect('equal')
axes[0, 0].grid(True, alpha=0.3)

total_log_det = np.zeros(len(z))
for i, layer in enumerate(flow_layers):
    z, ld = layer.forward(z)
    total_log_det += ld
    
    row = (i + 1) // 5
    col = (i + 1) % 5
    ax = axes[row, col]
    mask_type = 'fix x₁' if layer.mask_first else 'fix x₂'
    ax.scatter(z[:, 0], z[:, 1], s=1, alpha=0.3, c='coral')
    ax.set_title(f'Layer {i+1} ({mask_type})', fontsize=10)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

# Last subplot: show the final result
axes[1, 4].scatter(z[:, 0], z[:, 1], s=1, alpha=0.3, c='coral')
axes[1, 4].set_title('Final Output', fontsize=10)
axes[1, 4].set_aspect('equal')
axes[1, 4].grid(True, alpha=0.3)

plt.suptitle('Coupling Layers with Alternating Masks: Gaussian → Complex', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("🔑 Alternating masks ensure ALL dimensions get transformed.")
print("Each layer is simple and has O(n) Jacobian determinant.")
print("Together they can create arbitrarily complex distributions!")

In [ ]:
# Verify invertibility: map forward then backward

z_original = np.random.randn(1000, 2)

# Forward
z_forward = z_original.copy()
for layer in flow_layers:
    z_forward, _ = layer.forward(z_forward)

# Inverse (reverse order!)
z_reconstructed = z_forward.copy()
for layer in reversed(flow_layers):
    z_reconstructed = layer.inverse(z_reconstructed)

# Check reconstruction error
error = np.max(np.abs(z_original - z_reconstructed))
print(f"Maximum reconstruction error: {error:.2e}")
print(f"(Should be ~0, within floating point precision)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(z_original[:, 0], z_original[:, 1], s=5, alpha=0.5, c='steelblue')
axes[0].set_title('Original z₀')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(z_forward[:, 0], z_forward[:, 1], s=5, alpha=0.5, c='coral')
axes[1].set_title('After forward pass (x)')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

axes[2].scatter(z_reconstructed[:, 0], z_reconstructed[:, 1], s=5, alpha=0.5, c='green')
axes[2].set_title('After inverse pass (z₀ recovered)')
axes[2].set_aspect('equal')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Flow Invertibility: z₀ → x → z₀ (perfect reconstruction)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n🔑 Invertibility is EXACT — not approximate like VAE encoders!")
print("This means we can go from data → latent → data without any loss.")

## 5. Exact Likelihood — Why Flows Are Special

Normalizing flows can compute the **exact** log-likelihood of any data point:

$$\log p(\mathbf{x}) = \log p_0(f^{-1}(\mathbf{x})) + \log\left|\det\left(\frac{\partial f^{-1}}{\partial \mathbf{x}}\right)\right|$$

Compare with other generative models:

| Model | Likelihood | Type |
|-------|-----------|------|
| **Normalizing Flow** | $\log p(x)$ exact | **Exact** |
| VAE | $\text{ELBO} \leq \log p(x)$ | Lower bound |
| GAN | Not available | None |
| Diffusion | Requires many steps | Approximate |

This makes flows ideal for **density estimation** and **anomaly detection**.

In [ ]:
# Demonstrating exact likelihood computation

# Create a known target distribution: mixture of Gaussians
def sample_target(n):
    """Mixture of 3 Gaussians in 2D."""
    centers = [(-2, 0), (2, 0), (0, 2)]
    k = np.random.randint(0, 3, n)
    samples = np.array([centers[ki] for ki in k], dtype=float)
    samples += np.random.randn(n, 2) * 0.4
    return samples

# The flow maps data -> latent (inverse direction)
# log p(x) = log p_0(z) + sum(log_det)

def compute_flow_log_prob(x, layers):
    """Compute exact log probability under the flow."""
    z = x.copy()
    total_log_det = np.zeros(len(x))
    
    for layer in layers:
        z, log_det = layer.forward(z)
        total_log_det += log_det
    
    # Base distribution log probability
    log_p0 = -0.5 * np.sum(z**2, axis=1) - np.log(2 * np.pi)
    
    return log_p0 + total_log_det

# Evaluate on a grid
xx, yy = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-3, 4, 100))
grid_pts = np.column_stack([xx.ravel(), yy.ravel()])

log_probs = compute_flow_log_prob(grid_pts, flow_layers)
log_probs = log_probs.reshape(xx.shape)

# Evaluate specific points
test_points = np.array([[0, 0], [1, 1], [-2, 0], [3, 3]])
test_log_probs = compute_flow_log_prob(test_points, flow_layers)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Density contour
im = axes[0].contourf(xx, yy, np.exp(np.clip(log_probs, -20, 2)), levels=30, cmap='viridis')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Flow Density p(x) — Exact!')
axes[0].set_xlabel('x₁')
axes[0].set_ylabel('x₂')
axes[0].grid(True, alpha=0.3)

# Log-likelihood surface
im2 = axes[1].contourf(xx, yy, np.clip(log_probs, -10, 2), levels=30, cmap='plasma')
plt.colorbar(im2, ax=axes[1])
for pt, lp in zip(test_points, test_log_probs):
    axes[1].plot(pt[0], pt[1], 'w*', markersize=15)
    axes[1].annotate(f'log p={lp:.2f}', (pt[0]+0.1, pt[1]+0.1), color='white', fontsize=9)
axes[1].set_title('Log-likelihood log p(x)')
axes[1].set_xlabel('x₁')
axes[1].set_ylabel('x₂')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🔑 Key Insight: We can evaluate the EXACT probability of ANY point!")
print("No approximation, no bounds — just the actual probability density.")
print("\nThis is impossible with GANs and only approximate with VAEs.")
for pt, lp in zip(test_points, test_log_probs):
    print(f"  p({pt}) = exp({lp:.2f}) = {np.exp(lp):.6f}")

## 6. Training a Flow on Real Data

Training is simple: **maximize the log-likelihood** of the data.

$$\mathcal{L}(\theta) = \frac{1}{N} \sum_{i=1}^N \log p_\theta(x_i)$$

This is equivalent to minimizing $D_{KL}(p_{data} \| p_\theta)$ — the **forward KL divergence**.

Unlike GANs (which minimize reverse KL and are mode-seeking), flows minimize forward KL and are **mode-covering** — they try to place probability everywhere the data exists.

In [ ]:
# Training a normalizing flow on 2D "two moons" data

np.random.seed(42)

# Generate two-moons data
def make_moons(n, noise=0.08):
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.column_stack([np.cos(t), np.sin(t)])
    x2 = np.column_stack([1 - np.cos(t), -np.sin(t) + 0.5])
    return np.vstack([x1, x2]) + np.random.randn(n, 2) * noise

data = make_moons(2000)

# Build trainable flow with coupling layers
class TrainableCoupling:
    """Coupling layer with trainable parameters."""
    def __init__(self, mask_dim, hidden=8):
        self.mask_dim = mask_dim
        self.h = hidden
        # Scale network: input_dim=1 -> hidden -> 1
        self.Ws1 = np.random.randn(1, hidden) * 0.3
        self.bs1 = np.zeros(hidden)
        self.Ws2 = np.random.randn(hidden, 1) * 0.05
        self.bs2 = np.zeros(1)
        # Translate network
        self.Wt1 = np.random.randn(1, hidden) * 0.3
        self.bt1 = np.zeros(hidden)
        self.Wt2 = np.random.randn(hidden, 1) * 0.05
        self.bt2 = np.zeros(1)
    
    def _compute_st(self, x_cond):
        hs = np.tanh(x_cond @ self.Ws1 + self.bs1)
        s = np.clip(hs @ self.Ws2 + self.bs2, -3, 3)
        ht = np.tanh(x_cond @ self.Wt1 + self.bt1)
        t = ht @ self.Wt2 + self.bt2
        return s, t
    
    def forward(self, z):
        cond_dim = self.mask_dim
        trans_dim = 1 - self.mask_dim
        s, t = self._compute_st(z[:, cond_dim:cond_dim+1])
        y = z.copy()
        y[:, trans_dim:trans_dim+1] = z[:, trans_dim:trans_dim+1] * np.exp(s) + t
        return y, s.flatten()
    
    def inverse(self, y):
        cond_dim = self.mask_dim
        trans_dim = 1 - self.mask_dim
        s, t = self._compute_st(y[:, cond_dim:cond_dim+1])
        x = y.copy()
        x[:, trans_dim:trans_dim+1] = (y[:, trans_dim:trans_dim+1] - t) * np.exp(-s)
        return x
    
    def get_params(self):
        return [self.Ws1, self.bs1, self.Ws2, self.bs2,
                self.Wt1, self.bt1, self.Wt2, self.bt2]

# Create flow
n_flow_layers = 6
train_layers = [TrainableCoupling(mask_dim=i % 2, hidden=8) for i in range(n_flow_layers)]

# Training loop with numerical gradients (simplified)
lr = 0.005
batch_size = 200
losses = []

for step in range(600):
    idx = np.random.randint(0, len(data), batch_size)
    batch = data[idx]
    
    # Forward: data -> latent
    z = batch.copy()
    total_ld = np.zeros(batch_size)
    for layer in train_layers:
        z, ld = layer.forward(z)
        total_ld += ld
    
    log_p0 = -0.5 * np.sum(z**2, axis=1) - np.log(2 * np.pi)
    log_px = log_p0 + total_ld
    loss = -np.mean(log_px)
    losses.append(loss)
    
    # Numerical gradient update
    eps = 5e-4
    for layer in train_layers:
        for param in layer.get_params():
            flat = param.ravel()
            for j in range(len(flat)):
                flat[j] += eps
                z2 = batch.copy()
                tld2 = np.zeros(batch_size)
                for l in train_layers:
                    z2, ld2 = l.forward(z2)
                    tld2 += ld2
                lp2 = -0.5 * np.sum(z2**2, axis=1) - np.log(2*np.pi)
                loss_plus = -np.mean(lp2 + tld2)
                
                flat[j] -= 2 * eps
                z2 = batch.copy()
                tld2 = np.zeros(batch_size)
                for l in train_layers:
                    z2, ld2 = l.forward(z2)
                    tld2 += ld2
                lp2 = -0.5 * np.sum(z2**2, axis=1) - np.log(2*np.pi)
                loss_minus = -np.mean(lp2 + tld2)
                
                flat[j] += eps  # restore
                grad = (loss_plus - loss_minus) / (2 * eps)
                flat[j] -= lr * grad
    
    if step % 100 == 0:
        print(f"Step {step}: NLL = {loss:.4f}")

print("Training complete!")

In [ ]:
# Visualize trained flow results

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. Generate samples
z_gen = np.random.randn(2000, 2)
x_gen = z_gen.copy()
for layer in reversed(train_layers):
    x_gen = layer.inverse(x_gen)

axes[0].scatter(data[:, 0], data[:, 1], s=3, alpha=0.3, c='steelblue', label='Real')
axes[0].scatter(x_gen[:, 0], x_gen[:, 1], s=3, alpha=0.3, c='coral', label='Generated')
axes[0].set_title('Samples: Real vs Generated')
axes[0].legend()
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# 2. Density contour
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 100), np.linspace(-1, 2, 100))
grid_pts = np.column_stack([xx.ravel(), yy.ravel()])
z_grid = grid_pts.copy()
tld_grid = np.zeros(len(grid_pts))
for layer in train_layers:
    z_grid, ld = layer.forward(z_grid)
    tld_grid += ld
lp_grid = -0.5 * np.sum(z_grid**2, axis=1) - np.log(2*np.pi) + tld_grid

axes[1].contourf(xx, yy, np.exp(np.clip(lp_grid.reshape(xx.shape), -10, 3)), 
                 levels=30, cmap='viridis')
axes[1].scatter(data[:, 0], data[:, 1], s=1, alpha=0.2, c='white')
axes[1].set_title('Learned Density')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

# 3. Latent space
z_mapped = data[:500].copy()
for layer in train_layers:
    z_mapped, _ = layer.forward(z_mapped)
    
axes[2].scatter(z_mapped[:, 0], z_mapped[:, 1], s=3, alpha=0.5, c='coral')
theta = np.linspace(0, 2*np.pi, 100)
for r in [1, 2, 3]:
    axes[2].plot(r*np.cos(theta), r*np.sin(theta), 'k--', alpha=0.2)
axes[2].set_title('Latent Space (should be Gaussian)')
axes[2].set_aspect('equal')
axes[2].grid(True, alpha=0.3)

# 4. Training loss
axes[3].plot(losses, 'b-', alpha=0.7)
axes[3].set_xlabel('Step')
axes[3].set_ylabel('Negative Log-Likelihood')
axes[3].set_title('Training Loss')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🔑 Summary:")
print("• The flow learns to generate samples matching the data distribution")
print("• Density evaluation is exact — shown as contour plot")
print("• Latent space is approximately Gaussian (as intended)")
print("• Training is stable — just minimize negative log-likelihood!")

## 7. The Complete Picture

### How Normalizing Flows Fit in the Generative Model Landscape

| Feature | Flows | GANs | VAEs | Diffusion |
|---------|-------|------|------|-----------|
| **Likelihood** | Exact | None | ELBO | Approximate |
| **Sample quality** | Good | Excellent | Fair | Excellent |
| **Training** | Stable (MLE) | Unstable | Stable | Stable |
| **Architecture** | Must be invertible | Free | Free | Free |
| **Latent space** | Invertible, meaningful | None | Approximate | None |

### Key Takeaways

1. **Change of variables** lets us track how probability transforms — exactly
2. **The Jacobian determinant** measures local stretching/compression
3. **Coupling layers** give us $O(n)$ Jacobian and easy inversion
4. **Composition** of simple transforms creates complex distributions
5. **Exact likelihood** enables density estimation, anomaly detection, and model comparison
6. The trade-off: flows require **invertible architecture** (constraining model design)

### What's Next?

In Tutorial 18 (Flow Matching), we'll see how to combine the ideas of normalizing flows with
continuous-time dynamics, removing the architectural constraints while keeping the power of
learning probability flows.